In [48]:
#석식의 경우 수요일과 금요일은 한 건물에서만 한끼를 제공하는데 이 건물도 무작위로 바뀜
#첫번째 행이 NAN로 시작한다면 식단이 없는 것으로 간주

#디폴트는 세가지 한식,일품,웰빙 세가지를 모두 작성하는 것
#본관 3개 식당(월화목) 1개 식당(수금)
#C 3개 식당(월화목)
#C1 1개 식당(월화목)
#A1 1개 식당(월화목)
#A2 1개 식당(월화목)

#주말 식단은 특정파일을 바꿔가면서 식단 작성
#그래서 따로 작성 

In [49]:
#한 셀씩 실행하면서 확인하려면 해당 셀 지정 후 ctrl+enter

In [50]:
import win32com.client as win32
from contextlib import suppress

import shutil
from datetime import date, datetime
import os

In [51]:
#연도 바뀌면 연도부분 경로 수정필요

In [52]:
os.getcwd()

'D:\\Program_Files\\SNJ\\현대그린푸드'

In [105]:
import win32com.client as win32
from contextlib import suppress
import shutil
from datetime import date, datetime, timedelta
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np

#올해는 몇년도인가요?#########################################################(직접 작성)
now_y="25년"

origin_str = (datetime.now() - timedelta(weeks=2) - timedelta(days=1)).strftime("%m.%d")
#임시로 고정
origin_str = '2차서식'

#임시로 고정 ###########################################직접 수정
now_str = input("작업할 날짜를 넣어주세요")#"09.02"
print("작업할 파일의 날짜:",now_str)


# 이번달 
now_m = str(int(now_str[:2]))+"월"

source_folder = r".\{0}\{1}\{2}.{3}".format(now_y, now_m, str(int(now_str[:2])),str(int(now_str[3:])))  #메인경로 수정
print("가져올 폴더:",source_folder)

작업할 날짜를 넣어주세요 12.25


작업할 파일의 날짜: 12.25
가져올 폴더: .\25년\12월\12.25


In [106]:
#작업할 요일에 따라 컬럼이 달라짐
today = input("요일을 넣어주세요")
col_name = {"월":"Unnamed: 5", "화":"Unnamed: 6", "수":"Unnamed: 7", "목":"Unnamed: 8", "금":"Unnamed: 9", "토":"Unnamed: 10", "일":"Unnamed: 11"}

요일을 넣어주세요 목


In [107]:
now_d=str(int(now_str[:2]))+"."+str(int(now_str[3:]))
now_d2 = now_d.replace(".","/")
print("서식변경한 오늘 날짜:",now_d2)

서식변경한 오늘 날짜: 12/25


In [108]:
#주간식단 파일 위치 (매주 수동변경)
week_meal = "2025.12.22.현대차남양.통합주간식단표(게시)_본관.csv"

In [110]:
import pandas as pd
df=pd.read_csv("./"+week_meal, encoding='utf-8')#'cp949' 'euc-kr' 'cp949'
display(df)

,12,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,NaN,NaN,NaN,NaN,NaN,871kcal/112:27:35,403kcal/37:23:18,NaN,NaN,NaN,NaN,NaN
126,NaN,NaN,후 식,NaN,NaN,옥수수보리차,옥수수보리차,NaN,NaN,NaN,NaN,NaN


In [111]:
df.loc[105:111,col_name[today]] #한식
df.loc[113:119,col_name[today]] #일품
df.loc[121:125,col_name[today]] #웰빙스낵

121    NaN
122    NaN
123    NaN
124    NaN
125    NaN
Name: Unnamed: 8, dtype: object

In [112]:
#8행에 요일정보가 포함됨
#월:5번째 열부터 목:8번째 열까지 
df.iloc[8:,5]

#석식은 105부터 125행까지
df[105:126]

#본관 조식
#df.loc[105:111,col_name[today]] #한식
#메인메뉴
main_menu1 = next((x for x in df.loc[105:111,col_name[today]] if x.startswith("★")), None)
main_menu1_1=main_menu1.replace("★","")
print(main_menu1_1)
#그외 메뉴들 (메인에 들어가면 안되는 반찬 제외)
#공백은 float이라서 문자열을 처리하는데 오류가 생김 따라서 str형 데이터만 처리할 수 있도록 조건 추가 
other_menu1 = [x.replace("★","") for x in df.loc[105:110,col_name[today]] if (x not in [main_menu1, main_menu1_1,"깍두기", "누룽지숭늉", "배추김치", "수제배추겉절이", "포기김치"]) & (type(x)==str)]
print(other_menu1)
#칼로리정보
menu_data1 = df.loc[111,col_name[today]]
print(menu_data1)

# df.loc[113:119,col_name[today]] #일품
#메인메뉴
main_menu2 = next((x for x in df.loc[113:119,col_name[today]] if x.startswith("★")), None)
main_menu2_1=main_menu2.replace("★","")
print(main_menu2_1)
#그외 메뉴들 (메인에 들어가면 안되는 반찬 제외)
#공백은 float이라서 문자열을 처리하는데 오류가 생김 따라서 str형 데이터만 처리할 수 있도록 조건 추가 
other_menu2 = [x.replace("★","") for x in df.loc[113:118,col_name[today]] if (x not in [main_menu2, main_menu2_1, "깍두기", "누룽지숭늉", "배추김치", "수제배추겉절이", "포기김치"]) & (type(x)==str)]
print(other_menu2)
#칼로리정보
menu_data2 = df.loc[119,col_name[today]]
print(menu_data2)

# df.loc[121:125,col_name[today]] #웰빙스낵
#메인메뉴
main_menu3 = next((x for x in df.loc[121:125,col_name[today]] if x.startswith("★")), None)
main_menu3_1 = main_menu3.replace("★","")
print(main_menu3_1)
#그외 메뉴들 (메인에 들어가면 안되는 반찬 제외)
#공백은 float이라서 문자열을 처리하는데 오류가 생김 따라서 str형 데이터만 처리할 수 있도록 조건 추가 
other_menu3 = [x.replace("★","") for x in df.loc[121:124, col_name[today]] if (x not in [main_menu3, main_menu3_1, "깍두기", "누룽지숭늉", "배추김치", "수제배추겉절이", "포기김치"]) & (type(x)==str)]
print(other_menu3)
#칼로리정보
menu_data3 = df.loc[125,col_name[today]]
print(menu_data3)

사천식자장덮밥
['청파탕', '고추잡채&꽃빵', '중국식오이무침']
1122kcal/156:39:38


AttributeError: 'float' object has no attribute 'startswith'

In [ ]:
#파워포인트 디렉토리
root_path=os.getcwd()
print(root_path) #root 디렉토리
folder_path = root_path+"\\25년"+"\\"+now_m+"\\"+now_d+"\\" #폴더 디렉토리 ####################수정
print(folder_path)

#root파일명 
#메인 조식(건물명) 날짜 / 메인 중식(건물명) 날짜 / 메인 석식(건물명) 날짜 
#ex)메인 조식(본관) 8.25.pptx
file_name = "메인 조식" + "(본관) " + now_d + ".pptx"  #가운데 건물명 변경하면서 작업 
print("확인용)",file_name)

#코너 파일명
#코너 조식 날짜 원산지 입력
#ex)코너 조식 8.25 원산지 입력
today_corner = f"코너 조식 {now_d} 원산지 입력.pptx"
print(today_corner)

In [ ]:
#ppt 파일 열기
import win32com.client

file_name = f'메인 석식(본관) {now_d}.pptx'

#powerpoint = win32com.client.Dispatch('PowerPoint.Application') #읽기모드 (이러면 수정은 되지만 현재 파일에 덮어쓰는게 안됨)
#powerpoint.Visible = True #PPT창 보이게 하기 

#파워포인트 프레젠테이션 불러오기
powerpoint = win32.gencache.EnsureDispatch("PowerPoint.Application") #읽기쓰기 모드로 열어야 현재 파일에 덮어쓰기 가능
#기존 파일 불러오기 예시
#presentation = powerpoint.Presentations.Open(str(match[0].resolve(), WithWindow=False))###파일 열때 파일명 앞에 절대경로 수정 필요 

#본관 조식 작업
presentation = powerpoint.Presentations.Open(folder_path+file_name, ReadOnly=False, WithWindow=True) #WithWindow=False로 하면 화면 안열고 진행 

In [ ]:
#메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
#" / ".join(other_menu3)

In [ ]:
#본관 석식 2번째 슬라이드
slide = presentation.Slides(2)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 8 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu3_1

#반찬 수정
# shape_index = 9 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu3)


In [ ]:
#본관 조식 4번째 슬라이드
slide = presentation.Slides(3)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow
#한그릇정식
#메인메뉴 데이터 수정
shape_index = 7 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu1_1
#반찬 수정
# shape_index = 9 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu1)

#누들
#메인메뉴 데이터 수정
shape_index = 4 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu2_1
#반찬 수정
# shape_index = 5 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu2)

In [ ]:
#본관 석식 4번째 슬라이드
slide = presentation.Slides(4)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한그릇정식
#메인메뉴 데이터 수정
shape_index = 11 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu1_1
#반찬 수정
# shape_index = 12 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu1)

In [ ]:
# 같은 파일에 덮어쓰기
presentation.Save()
#파일 옵션에 의해 기존 파일에 저장이 안되기 때문에 다른 이름으로 저장

#presentation.SaveAs(folder_path + "작업완료\\" + file_name)

# 닫기
presentation.Close()
#Quit는 Presentation(프레젠테이션) 객체가 아니라 Application(파워포인트 앱) 객체의 메서드라서 powerpoint를 닫아줘야함. 
#powerpoint.Quit()

In [ ]:
#메인 조식 C지구

#ppt 파일 열기
import win32com.client
file_name = f'메인 석식(C지구) {now_d}.pptx'
#파워포인트 프레젠테이션 불러오기
powerpoint = win32.gencache.EnsureDispatch("PowerPoint.Application") #읽기쓰기 모드로 열어야 현재 파일에 덮어쓰기 가능
#기존 파일 불러오기 예시
#presentation = powerpoint.Presentations.Open(str(match[0].resolve(), WithWindow=False))###파일 열때 파일명 앞에 절대경로 수정 필요 
#본관 조식 작업
presentation = powerpoint.Presentations.Open(folder_path+file_name, ReadOnly=False, WithWindow=True) #WithWindow=False로 하면 화면 안열고 진행 

In [ ]:
#본관 석식 2번째 슬라이드
slide = presentation.Slides(2)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 8 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu1_1

#반찬 수정
# shape_index = 9 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = "/ ".join(other_menu1)

#본관 석식 3번째 슬라이드
slide = presentation.Slides(3)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 8 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu3_1

#반찬 수정
# shape_index = 9 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu3)

#본관 석식 4번째 슬라이드
slide = presentation.Slides(4)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 7 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu2_1

#반찬 수정
# shape_index = 9 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu2)

In [ ]:
# 같은 파일에 덮어쓰기
presentation.Save()
#파일 옵션에 의해 기존 파일에 저장이 안되기 때문에 다른 이름으로 저장

#presentation.SaveAs(folder_path + "작업완료\\" + file_name)

# 닫기
presentation.Close()
#Quit는 Presentation(프레젠테이션) 객체가 아니라 Application(파워포인트 앱) 객체의 메서드라서 powerpoint를 닫아줘야함. 
#powerpoint.Quit()

In [ ]:
#주간식단 파일 위치 (매주 수동변경)
week_meal = "2025.12.22.현대차남양.통합주간식단표(게시)_C1.csv" #####################파일 변경
df=pd.read_csv("./"+week_meal, encoding='utf-8')#'cp949' 'euc-kr' 'cp949'
df[72:78]

In [ ]:
#메인메뉴
main_menu4 = next((x for x in df.loc[72:78,col_name[today]] if x.startswith("★")), None)
main_menu4_1 = main_menu4.replace("★","")

#그외 메뉴들 (메인에 들어가면 안되는 반찬 제외)
#공백은 float이라서 문자열을 처리하는데 오류가 생김 따라서 str형 데이터만 처리할 수 있도록 조건 추가 
other_menu4 = [x.replace("★","") for x in df.loc[72:77, col_name[today]] if (x not in [main_menu4, main_menu4_1, "깍두기", "누룽지숭늉", "배추김치", "수제배추겉절이", "포기김치"]) & (type(x)==str)]

print(main_menu4_1, other_menu4)

In [ ]:
#메인 석식 C1 지구

#ppt 파일 열기
import win32com.client
file_name = f'메인 석식(씨원) {now_d}.pptx'
#파워포인트 프레젠테이션 불러오기
powerpoint = win32.gencache.EnsureDispatch("PowerPoint.Application") #읽기쓰기 모드로 열어야 현재 파일에 덮어쓰기 가능
#기존 파일 불러오기 예시
#presentation = powerpoint.Presentations.Open(str(match[0].resolve(), WithWindow=False))###파일 열때 파일명 앞에 절대경로 수정 필요 
#본관 조식 작업
presentation = powerpoint.Presentations.Open(folder_path+file_name, ReadOnly=False, WithWindow=True) #WithWindow=False로 하면 화면 안열고 진행 

In [ ]:
#C1 석식 1번째 슬라이드
slide = presentation.Slides(2)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 9 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu4_1
#반찬 수정
# shape_index = 10 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu4)

In [ ]:
# 같은 파일에 덮어쓰기
presentation.Save()
# 닫기
presentation.Close()
#Quit는 Presentation(프레젠테이션) 객체가 아니라 Application(파워포인트 앱) 객체의 메서드라서 powerpoint를 닫아줘야함. 
#powerpoint.Quit()

In [ ]:
#주간식단 파일 위치 (매주 수동변경)
week_meal = "2025.12.22.현대차남양.통합주간식단표(게시)_A1.csv" #####################파일 변경
df=pd.read_csv("./"+week_meal, encoding='utf-8')#'cp949' 'euc-kr' 'cp949'
df[66:72]

In [ ]:
#메인메뉴
main_menu5 = next((x for x in df.loc[66:72,col_name[today]] if x.startswith("★")), None)
main_menu5_1 = main_menu5.replace("★","")

#그외 메뉴들 (메인에 들어가면 안되는 반찬 제외)
#공백은 float이라서 문자열을 처리하는데 오류가 생김 따라서 str형 데이터만 처리할 수 있도록 조건 추가 
other_menu5 = [x.replace("★","") for x in df.loc[66:71, col_name[today]] if (x not in [main_menu5, main_menu5_1, "깍두기", "누룽지숭늉", "배추김치", "수제배추겉절이", "포기김치"]) & (type(x)==str)]

print(main_menu5_1, other_menu5)

In [ ]:
#메인 조식 A1(디자인)지구

#ppt 파일 열기
import win32com.client
file_name = f'메인 석식(디자인) {now_d}.pptx'
#파워포인트 프레젠테이션 불러오기
powerpoint = win32.gencache.EnsureDispatch("PowerPoint.Application") #읽기쓰기 모드로 열어야 현재 파일에 덮어쓰기 가능
#기존 파일 불러오기 예시
#presentation = powerpoint.Presentations.Open(str(match[0].resolve(), WithWindow=False))###파일 열때 파일명 앞에 절대경로 수정 필요 
#본관 조식 작업
presentation = powerpoint.Presentations.Open(folder_path+file_name, ReadOnly=False, WithWindow=True) #WithWindow=False로 하면 화면 안열고 진행 

In [ ]:
#A1 석식 1번째 슬라이드
slide = presentation.Slides(2)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 9 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu5_1
#반찬 수정
# shape_index = 10 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu5)


In [ ]:
# 같은 파일에 덮어쓰기
presentation.Save()
# 닫기
presentation.Close()
#Quit는 Presentation(프레젠테이션) 객체가 아니라 Application(파워포인트 앱) 객체의 메서드라서 powerpoint를 닫아줘야함. 
#powerpoint.Quit()

In [ ]:
#주간식단 파일 위치 (매주 수동변경)
week_meal = "2025.12.22.현대차남양.통합주간식단표(게시)_A2.csv" #####################파일 변경
df=pd.read_csv("./"+week_meal, encoding='utf-8')#'cp949' 'euc-kr' 'cp949'
df[74:80]

In [ ]:
df.loc[74:80,col_name[today]]

In [ ]:
#메인메뉴
main_menu6 = next((x for x in df.loc[74:79,col_name[today]] if x.startswith("★")), None)
main_menu6_1 = main_menu6.replace("★","")

#그외 메뉴들 (메인에 들어가면 안되는 반찬 제외)
#공백은 float이라서 문자열을 처리하는데 오류가 생김 따라서 str형 데이터만 처리할 수 있도록 조건 추가 
other_menu6 = [x.replace("★","") for x in df.loc[74:79, col_name[today]] if (x not in [main_menu6, main_menu6_1, "깍두기", "누룽지숭늉", "배추김치", "수제배추겉절이", "포기김치"]) & (type(x)==str)]

print(main_menu6_1, other_menu6)

In [ ]:
#메인 조식 A2(파워포인트)지구

#ppt 파일 열기
import win32com.client
file_name = f'메인 석식(파워트레인) {now_d}.pptx'
#파워포인트 프레젠테이션 불러오기
powerpoint = win32.gencache.EnsureDispatch("PowerPoint.Application") #읽기쓰기 모드로 열어야 현재 파일에 덮어쓰기 가능
#기존 파일 불러오기 예시
#presentation = powerpoint.Presentations.Open(str(match[0].resolve(), WithWindow=False))###파일 열때 파일명 앞에 절대경로 수정 필요 
#본관 조식 작업
presentation = powerpoint.Presentations.Open(folder_path+file_name, ReadOnly=False, WithWindow=True) #WithWindow=False로 하면 화면 안열고 진행 

In [ ]:
#A2 석식 1번째 슬라이드
slide = presentation.Slides(2)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 7 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu6_1
#반찬 수정
# shape_index = 8 # 수정할 텍스트박스의 Shape 번호
# shape = slide.Shapes(shape_index)
# #메인메뉴 말고 반찬은 기존에 리스트로 되어있어서 . 기준으로 join해줌
# shape.TextFrame.TextRange.Text = " / ".join(other_menu6)

In [ ]:
# 같은 파일에 덮어쓰기
presentation.Save()
# 닫기
presentation.Close()
#Quit는 Presentation(프레젠테이션) 객체가 아니라 Application(파워포인트 앱) 객체의 메서드라서 powerpoint를 닫아줘야함. 
#powerpoint.Quit()

In [ ]:
#코너

In [ ]:
#석식 코너

#ppt 파일 열기
import win32com.client
file_name = f'코너 석식 {now_d} 원산지 입력.pptx'
#파워포인트 프레젠테이션 불러오기
powerpoint = win32.gencache.EnsureDispatch("PowerPoint.Application") #읽기쓰기 모드로 열어야 현재 파일에 덮어쓰기 가능
#기존 파일 불러오기 예시
#presentation = powerpoint.Presentations.Open(str(match[0].resolve(), WithWindow=False))###파일 열때 파일명 앞에 절대경로 수정 필요 
#본관 조식 작업
presentation = powerpoint.Presentations.Open(folder_path+file_name, ReadOnly=False, WithWindow=True) #WithWindow=False로 하면 화면 안열고 진행 

In [ ]:
#주간식단 파일 위치 (매주 수동변경)
week_meal = "2025.12.22.현대차남양.통합주간식단표(게시)_본관.csv" #####################파일 변경
df=pd.read_csv("./"+week_meal, encoding='UTF-8')#'cp949' 'euc-kr' 'cp949'
# df[105:112]
# df[113:120]
# df[121:126]

In [ ]:
slide = presentation.Slides(1)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 6 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu1_1

#메인메뉴 원산지 수정
#반찬 수정
shape_index = 5 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)

#기존메뉴정보에는 빠진 반찬들이 있어 따로 처리
other_menu1_1 = [x.replace("★","") for x in df.loc[105:110, col_name[today]] if x not in [main_menu1, main_menu1_1] if type(x)==str]
other_menu1_1.insert(0, main_menu1_1)

other_menu1_1.insert(1, "\n")
other_menu1_1.insert(3, "\n")
other_menu1_1.insert(5, "\n")
other_menu1_1.insert(7, "\n")
other_menu1_1.insert(9, "\n")
other_menu1_1.insert(11, "\n")
other_menu1_1.insert(13, "\n")
other_menu1_1

#print("".join(other_menu1_1))
shape.TextFrame.TextRange.Text = "".join(other_menu1_1)

#한식 탄단지
shape_index = 8 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)

parts = re.split(r"[/:]", menu_data1) #/랑 :기준으로 split
parts.insert(1, " (탄 ")
parts.insert(3, "g/")
parts.insert(4, "단 ")
parts.insert(6, "g/")
parts.insert(7, "지 ")
parts.insert(9, "g)")

parts="".join(parts)
shape.TextFrame.TextRange.Text = parts

In [ ]:
#작업을 위한 셀 #################################################################################################작업용
#현재 선택된 개체 확인(확인용)
# selection = window.Selection
# selection

slide = presentation.Slides(2)
for i, shape in enumerate(slide.Shapes):
    #모든 요소는 몇번째인지로 지정해서 수정이 들어감 ############################################################################
    print(f"{i+1}번째 요소입니다.") #파이썬은 0부터 세고 ppt의 요소는 1부터 셈
    #print("이름", shape.Name)
    #print("타입", shape.Type)

        # 텍스트 있는 경우만 내용 출력
    if getattr(shape, "HasTextFrame", 0):
        if shape.TextFrame.HasText:
            text = shape.TextFrame.TextRange.Text
            text = text.replace("\r", " ").replace("\n", " ")  # 줄바꿈 제거
            print("텍스트 내용 :", text)
        else:
            print("텍스트 내용 : (비어 있음)")
    else:
        print("텍스트 내용 : (텍스트 프레임 없음)")
#presentation.Slides(3).Select()


In [ ]:
slide = presentation.Slides(2)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#일품
#메인메뉴 데이터 수정
shape_index = 6 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu2_1

#메인메뉴 원산지 수정
#반찬 수정
shape_index = 5 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)

#기존메뉴정보에는 빠진 반찬들이 있어 따로 처리
other_menu2_1 = [x.replace("★","") for x in df.loc[113:118, col_name[today]] if x not in [main_menu2, main_menu2_1] if type(x)==str]
other_menu2_1.insert(0, main_menu2_1)

other_menu2_1.insert(1, "\n")
other_menu2_1.insert(3, "\n")
other_menu2_1.insert(5, "\n")
other_menu2_1.insert(7, "\n")
other_menu2_1.insert(9, "\n")
other_menu2_1.insert(11, "\n")
other_menu2_1.insert(13, "\n")
other_menu2_1

#print("".join(other_menu1_1))
shape.TextFrame.TextRange.Text = "".join(other_menu2_1)

#한식 탄단지
shape_index = 8 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)

parts = re.split(r"[/:]", menu_data2) #/랑 :기준으로 split
parts.insert(1, " (탄 ")
parts.insert(3, "g/")
parts.insert(4, "단 ")
parts.insert(6, "g/")
parts.insert(7, "지 ")
parts.insert(9, "g)")

parts="".join(parts)
shape.TextFrame.TextRange.Text = parts

In [ ]:
slide = presentation.Slides(3)
presentation = powerpoint.ActivePresentation
window = powerpoint.ActiveWindow

#한식
#메인메뉴 데이터 수정
shape_index = 2 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)
shape.TextFrame.TextRange.Text = main_menu3_1

#메인메뉴 원산지 수정
#반찬 수정
shape_index = 3 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)

#기존메뉴정보에는 빠진 반찬들이 있어 따로 처리
other_menu3_1 = [x.replace("★","") for x in df.loc[121:124, col_name[today]] if x not in [main_menu3, main_menu3_1] if type(x)==str]
other_menu3_1.insert(0, main_menu3_1)

other_menu3_1.insert(1, "\n")
other_menu3_1.insert(3, "\n")
other_menu3_1.insert(5, "\n")
other_menu3_1.insert(7, "\n")
other_menu3_1.insert(9, "\n")
other_menu3_1.insert(11, "\n")
other_menu3_1.insert(13, "\n")
other_menu3_1

#print("".join(other_menu1_1))
shape.TextFrame.TextRange.Text = "".join(other_menu3_1)

#한식 탄단지
shape_index = 4 # 수정할 텍스트박스의 Shape 번호
shape = slide.Shapes(shape_index)

parts = re.split(r"[/:]", menu_data3) #/랑 :기준으로 split
parts.insert(1, " (탄 ")
parts.insert(3, "g/")
parts.insert(4, "단 ")
parts.insert(6, "g/")
parts.insert(7, "지 ")
parts.insert(9, "g)")

parts="".join(parts)
shape.TextFrame.TextRange.Text = parts

In [ ]:
# 같은 파일에 덮어쓰기
presentation.Save()
# 닫기
presentation.Close()
#Quit는 Presentation(프레젠테이션) 객체가 아니라 Application(파워포인트 앱) 객체의 메서드라서 powerpoint를 닫아줘야함. 
powerpoint.Quit()

In [ ]:
# #작업을 위한 셀 #################################################################################################작업용
# #현재 선택된 개체 확인(확인용)
# # selection = window.Selection
# # selection

# slide = presentation.Slides(4)
# for i, shape in enumerate(slide.Shapes):
#     #모든 요소는 몇번째인지로 지정해서 수정이 들어감 ############################################################################
#     print(f"{i+1}번째 요소입니다.") #파이썬은 0부터 세고 ppt의 요소는 1부터 셈
#     #print("이름", shape.Name)
#     #print("타입", shape.Type)

#         # 텍스트 있는 경우만 내용 출력
#     if getattr(shape, "HasTextFrame", 0):
#         if shape.TextFrame.HasText:
#             text = shape.TextFrame.TextRange.Text
#             text = text.replace("\r", " ").replace("\n", " ")  # 줄바꿈 제거
#             print("텍스트 내용 :", text)
#         else:
#             print("텍스트 내용 : (비어 있음)")
#     else:
#         print("텍스트 내용 : (텍스트 프레임 없음)")
# #presentation.Slides(3).Select()


In [ ]:
################철야자##########################################################################################################

#철야자는 본관이나 C지구 등 랜덤으로 나와서 이건 수작업 